# 02 · Resultados principales

Este cuaderno **solo lee la evidencia versionada** en `reports/forecasting/evidence/` y la presenta en tablas. No recalcula ninguna cifra ni entrena modelos, y no escribe archivos. Cada tabla indica su archivo de origen.

- **Evaluación preregistrada** (receta congelada, prueba única 2023–2025): las conclusiones vinculantes del contrato (§8.4).
- **Análisis complementario** (eventos con la regla del producto B01, continuidad, M0 y regularización): posterior a la prueba y descriptivo; se documenta en `docs/forecasting/analisis_complementario.md`.

El análisis integrado de los resultados está en `docs/forecasting/analisis_resultados.md`.

In [1]:
import os
import sys
from pathlib import Path

# Rutas relativas a la raíz del repositorio, tanto si el cuaderno se abre desde notebooks/ como desde la raíz.
ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "forecasting.yaml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "configs" / "forecasting.yaml").is_file():
    raise RuntimeError("El cuaderno se ejecuta desde la raíz del repositorio o desde notebooks/.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import json

import pandas as pd

EVIDENCIA = Path("reports/forecasting/evidence")


def leer(relativa: str) -> dict:
    """JSON de la evidencia versionada; imprime su ruta como fuente de la tabla."""
    ruta = EVIDENCIA / relativa
    print(f"Fuente: {ruta.as_posix()}")
    return json.loads(ruta.read_text(encoding="utf-8"))


def de(n: int, total: int) -> str:
    return f"{n} de {total}"

## 1. Evaluación preregistrada en prueba (2023–2025)

Receta: regresión logística L2 por municipio con seis variables sin clima, τ = 0,05, k = 2 y W = 12 (`configs/receta_congelada.yaml`, hash `de74d4ab…fda5`). 936 filas (156 por municipio), con reentrenamiento anual de origen móvil. Menor es mejor en Brier y log-loss; mayor es mejor en AP.

In [3]:
prueba = leer("prueba_20261112/resumen_prueba.json")
est = prueba["estimacion"]
sistemas = {"modelo": "Receta (logística L2, sin clima)", "B2": "B2: zona actual del canal",
            "B1": "B1: estacionalidad de la semana objetivo", "ablacion": "Ablación: misma receta con clima"}
metricas = pd.DataFrame({etiqueta: {"Brier": est["brier"][s], "Log-loss": est["logloss"][s], "AP": est["ap"][s]}
                         for s, etiqueta in sistemas.items()}).T
print(f"Filas: {prueba['filas']}; prevalencia de la zona epidémica en t + 8: {est['prevalencia']:.1%}")
metricas.round(4)

Fuente: reports/forecasting/evidence/prueba_20261112/resumen_prueba.json
Filas: 936; prevalencia de la zona epidémica en t + 8: 34.4%


,Brier,Log-loss,AP
"Receta (logística L2, sin clima)",0.1413,0.4409,0.7651
B2: zona actual del canal,0.1537,0.4708,0.6115
B1: estacionalidad de la semana objetivo,0.2658,0.7810,0.2712
Ablación: misma receta con clima,0.1434,0.4501,0.7572


In [4]:
habilidad = pd.DataFrame({
    "frente a B2": {"BSS (1 − Brier / Brier de referencia)": est["bss_vs_b2"],
                    "Mejora relativa de log-loss": est["mejora_logloss_vs_b2"]},
    "frente a B1": {"BSS (1 − Brier / Brier de referencia)": est["bss_vs_b1"],
                    "Mejora relativa de log-loss": est["mejora_logloss_vs_b1"]},
})
habilidad.round(4)

,frente a B2,frente a B1
BSS (1 − Brier / Brier de referencia),0.0805,0.4684
Mejora relativa de log-loss,0.0635,0.4355


**Niveles de conclusión** (contrato §8.4 y D9): por casillas municipio-año ganadas (18 casillas), con el nivel más bajo entre Brier y log-loss.

In [5]:
nombres_comparacion = {"vs_b2": "Receta frente a B2", "vs_b1": "Receta frente a B1",
                       "ablacion_clima": "Receta frente a la ablación con clima"}
niveles = pd.DataFrame([
    {"Comparación": nombres_comparacion[clave], "Nivel Brier": c["nivel_brier"], "Nivel log-loss": c["nivel_logloss"],
     "Nivel": c["nivel"], "Casillas ganadas (Brier)": de(c["casillas_ganadas_brier"], c["casillas"]),
     "Casillas ganadas (log-loss)": de(c["casillas_ganadas_logloss"], c["casillas"])}
    for clave, c in prueba["conclusiones"].items()
]).set_index("Comparación")
niveles

,Nivel Brier,Nivel log-loss,Nivel,Casillas ganadas (Brier),Casillas ganadas (log-loss)
Comparación,,,,,
Receta frente a B2,tendencia favorable,tendencia favorable,tendencia favorable,12 de 18,12 de 18
Receta frente a B1,mejora consistente,mejora consistente,mejora consistente,15 de 18,15 de 18
Receta frente a la ablación con clima,sin mejora,tendencia favorable,sin mejora,9 de 18,10 de 18


## 2. Eventos: regla preregistrada, regla del producto y canal

- **Regla preregistrada** (contrato §7.4): emisión positiva si p̂ ≥ τ; alerta tras k = 2 emisiones positivas consecutivas. Es la regla de la evaluación y la de las conclusiones vinculantes.
- **Regla del producto B01** (enmienda D10.1, DV-07): emisión positiva si p̂ ≥ τ **y** la zona observada en t no es epidemia. Es la que muestra la API; se formuló después de conocer la prueba, así que sus cifras de prueba no son confirmatorias.
- **Canal**: emisión positiva si la zona observada en t es alerta.

Primero, los eventos de la evaluación preregistrada tal como los registra la prueba única:

In [6]:
ev = prueba["eventos"]
eventos_preregistrados = pd.DataFrame({
    "Modelo (regla preregistrada)": {
        "Inicios avisados": de(ev["avisados_modelo"], ev["inicios_evaluables"]),
        "Entradas avisadas": de(ev["entradas"]["avisadas_modelo"], ev["entradas"]["evaluables"]),
        "Falsas alarmas": ev["falsas_alarmas"]["modelo"],
        "Falsas alarmas por municipio-año": round(ev["falsas_alarmas_por_municipio_anio"]["modelo"], 2),
        "Anticipación mediana (semanas)": ev["anticipacion_mediana_semanas"]["modelo"]},
    "Canal": {
        "Inicios avisados": de(ev["avisados_canal"], ev["inicios_evaluables"]),
        "Entradas avisadas": de(ev["entradas"]["avisadas_canal"], ev["entradas"]["evaluables"]),
        "Falsas alarmas": ev["falsas_alarmas"]["canal"],
        "Falsas alarmas por municipio-año": round(ev["falsas_alarmas_por_municipio_anio"]["canal"], 2),
        "Anticipación mediana (semanas)": ev["anticipacion_mediana_semanas"]["canal"]},
})
m = ev["mcnemar"]
print(f"Municipio-años: {ev['municipio_anios']}. McNemar de inicios (modelo frente a canal): "
      f"{m['discrepantes']} discrepantes, {m['a_favor_modelo']} a favor del modelo; nivel «{m['nivel']}».")
eventos_preregistrados

Municipio-años: 18. McNemar de inicios (modelo frente a canal): 6 discrepantes, 1 a favor del modelo; nivel «sin mejora».


,Modelo (regla preregistrada),Canal
Inicios avisados,4 de 10,8 de 10
Entradas avisadas,13 de 41,19 de 41
Falsas alarmas,18,30
Falsas alarmas por municipio-año,1.0,1.67
Anticipación mediana (semanas),4.5,1.5


Comparación de las tres reglas con las mismas predicciones, τ, k y W, en prueba y en desarrollo (fuera de año):

In [7]:
b01 = leer("complementario_20261120/b01_resumen.json")
reglas = {"regla_actual": "Regla preregistrada (p̂ ≥ τ)", "B01": "Regla del producto (B01)", "canal": "Canal"}
filas = []
for periodo in ("prueba", "desarrollo"):
    for clave, etiqueta in reglas.items():
        g = b01["periodos"][periodo]["global"][clave]
        filas.append({
            "Periodo": periodo, "Sistema": etiqueta,
            "Inicios avisados": de(g["inicios_avisados"], g["inicios_evaluables"]),
            "Entradas avisadas": de(g["entradas_avisadas"], g["entradas_evaluables"]),
            "Alertas (de continuidad)": f"{g['alertas']} ({g['alertas_continuidad']})",
            "Falsas alarmas": g["falsas_alarmas"],
            "Semanas bajo alerta": f"{g['semanas_bajo_alerta']} de {g['semanas']} ({g['fraccion_bajo_alerta']:.1%})",
            "Anticipación mediana: contrato / primera alerta":
                f"{g['anticipacion_mediana_contrato']} / {g['anticipacion_mediana_primera']}",
        })
pd.DataFrame(filas).set_index(["Periodo", "Sistema"])

Fuente: reports/forecasting/evidence/complementario_20261120/b01_resumen.json


Inicios avisados Entradas avisadas  \
Periodo    Sistema                                                           
prueba     Regla preregistrada (p̂ ≥ τ)          4 de 10          13 de 41   
           Regla del producto (B01)              9 de 10          24 de 41   
           Canal                                 8 de 10          19 de 41   
desarrollo Regla preregistrada (p̂ ≥ τ)          8 de 10          21 de 44   
           Regla del producto (B01)             10 de 10          38 de 44   
           Canal                                10 de 10          36 de 44   

                                        Alertas (de continuidad)  \
Periodo    Sistema                                                 
prueba     Regla preregistrada (p̂ ≥ τ)                   26 (4)   
           Regla del producto (B01)                       38 (0)   
           Canal                                          39 (0)   
desarrollo Regla preregistrada (p̂ ≥ τ)                   43 (5)   
           Regla del producto (B01)                       59 (0)   
           Canal                                          65 (0)   

                                         Falsas alarmas  Semanas bajo alerta  \
Periodo    Sistema                                                             
prueba     Regla preregistrada (p̂ ≥ τ)              18   565 de 936 (60.4%)   
           Regla del producto (B01)                  26   192 de 936 (20.5%)   
           Canal                                     30   393 de 936 (42.0%)   
desarrollo Regla preregistrada (p̂ ≥ τ)              24  586 de 1248 (47.0%)   
           Regla del producto (B01)                  37  244 de 1248 (19.6%)   
           Canal                                     48  406 de 1248 (32.5%)   

                                        Anticipación mediana: contrato / primera alerta  
Periodo    Sistema                                                                       
prueba     Regla preregistrada (p̂ ≥ τ)                                       4.5 / 7.5  
           Regla del producto (B01)                                           1.0 / 3.0  
           Canal                                                              1.5 / 2.0  
desarrollo Regla preregistrada (p̂ ≥ τ)                                       7.5 / 8.0  
           Regla del producto (B01)                                           2.0 / 8.0  
           Canal                                                              3.0 / 5.5

In [8]:
nombres_mcnemar = {"B01_vs_regla_actual": "B01 frente a la regla preregistrada", "B01_vs_canal": "B01 frente al canal",
                   "regla_actual_vs_canal": "Regla preregistrada frente al canal"}
pd.DataFrame([
    {"Periodo": periodo, "Comparación de inicios (McNemar, una cola)": nombres_mcnemar[clave],
     "Discrepantes": c["discrepantes"], "A favor": c["a_favor"], "En contra": c["en_contra"], "Nivel": c["nivel"]}
    for periodo in ("prueba", "desarrollo")
    for clave, c in b01["periodos"][periodo]["comparaciones"].items()
]).set_index(["Periodo", "Comparación de inicios (McNemar, una cola)"])

Discrepantes  A favor  \
Periodo    Comparación de inicios (McNemar, una cola)                          
prueba     B01 frente a la regla preregistrada                    5        5   
           B01 frente al canal                                    1        1   
           Regla preregistrada frente al canal                    6        1   
desarrollo B01 frente a la regla preregistrada                    2        2   
           B01 frente al canal                                    0        0   
           Regla preregistrada frente al canal                    2        0   

                                                       En contra  \
Periodo    Comparación de inicios (McNemar, una cola)              
prueba     B01 frente a la regla preregistrada                 0   
           B01 frente al canal                                 0   
           Regla preregistrada frente al canal                 5   
desarrollo B01 frente a la regla preregistrada                 0   
           B01 frente al canal                                 0   
           Regla preregistrada frente al canal                 2   

                                                                     Nivel  
Periodo    Comparación de inicios (McNemar, una cola)                       
prueba     B01 frente a la regla preregistrada          mejora consistente  
           B01 frente al canal                         tendencia favorable  
           Regla preregistrada frente al canal                  sin mejora  
desarrollo B01 frente a la regla preregistrada         tendencia favorable  
           B01 frente al canal                                  sin mejora  
           Regla preregistrada frente al canal                  sin mejora

## 3. Continuidad: semanas que ya están en zona epidémica

Calidad de la probabilidad de seguir en zona epidémica en t + 8, separada por la zona observada en t. Confiabilidad y resolución: descomposición de Murphy con 10 intervalos de igual frecuencia dentro de cada estrato (menor confiabilidad y mayor resolución son mejores). Casillas: municipio-años con al menos 5 semanas en el estrato.

In [9]:
cont = leer("continuidad_20261120/continuidad_resumen.json")
estratos = {"zona_epidemia": "zona_t = epidemia", "zona_no_epidemia": "zona_t ≠ epidemia"}
filas = []
for periodo in ("desarrollo", "prueba"):
    for clave, etiqueta in estratos.items():
        e = cont["periodos"][periodo]["estratos"][clave]
        r, b, c = e["receta"], e["B2"], e["casillas"]
        filas.append({
            "Periodo": periodo, "Estrato": etiqueta, "Filas (positivas)": f"{e['filas']} ({e['positivas']})",
            "Log-loss receta / B2": f"{r['logloss']:.4f} / {b['logloss']:.4f}",
            "Brier receta / B2": f"{r['brier']:.4f} / {b['brier']:.4f}",
            "BSS frente a B2": round(e["bss_vs_b2"], 3),
            "Confiabilidad receta / B2": f"{r['confiabilidad']:.4f} / {b['confiabilidad']:.4f}",
            "Resolución receta / B2": f"{r['resolucion']:.4f} / {b['resolucion']:.4f}",
            "Casillas ganadas Brier / log-loss":
                f"{c['ganadas_brier']} / {c['ganadas_logloss']} de {c['casillas']}",
        })
pd.DataFrame(filas).set_index(["Periodo", "Estrato"])

Fuente: reports/forecasting/evidence/continuidad_20261120/continuidad_resumen.json


Filas (positivas) Log-loss receta / B2  \
Periodo    Estrato                                                    
desarrollo zona_t = epidemia         310 (231)      0.5595 / 0.6950   
           zona_t ≠ epidemia          938 (88)      0.2335 / 0.2831   
prueba     zona_t = epidemia         340 (243)      0.5769 / 0.6422   
           zona_t ≠ epidemia          596 (79)      0.3633 / 0.3730   

                             Brier receta / B2  BSS frente a B2  \
Periodo    Estrato                                                
desarrollo zona_t = epidemia   0.1873 / 0.2503            0.252   
           zona_t ≠ epidemia   0.0725 / 0.0827            0.123   
prueba     zona_t = epidemia   0.1935 / 0.2240            0.136   
           zona_t ≠ epidemia   0.1115 / 0.1136            0.018   

                             Confiabilidad receta / B2 Resolución receta / B2  \
Periodo    Estrato                                                              
desarrollo zona_t = epidemia           0.0230 / 0.0919        0.0249 / 0.0302   
           zona_t ≠ epidemia           0.0026 / 0.0030        0.0152 / 0.0064   
prueba     zona_t = epidemia           0.0124 / 0.0469        0.0244 / 0.0264   
           zona_t ≠ epidemia           0.0063 / 0.0100        0.0131 / 0.0117   

                             Casillas ganadas Brier / log-loss  
Periodo    Estrato                                              
desarrollo zona_t = epidemia                       8 / 8 de 12  
           zona_t ≠ epidemia                     16 / 16 de 23  
prueba     zona_t = epidemia                      10 / 9 de 14  
           zona_t ≠ epidemia                       8 / 9 de 17

In [10]:
pd.DataFrame([
    {"Periodo": periodo, "Falsas alarmas de B01": f["falsas_alarmas_B01"],
     "Empiezan en las 13 semanas tras un episodio": f["empiezan_tras_episodio"]}
    for periodo in ("desarrollo", "prueba")
    for f in [cont["periodos"][periodo]["falsas_alarmas_B01"]]
]).set_index("Periodo")

,Falsas alarmas de B01,Empiezan en las 13 semanas tras un episodio
Periodo,,
desarrollo,37,13
prueba,26,11


## 4. M0: modelo mínimo con la posición frente al canal

M0 es una regresión logística L2 con `posicion_4s` como única variable, con el mismo preprocesamiento, rejilla de C y procedimiento de selección que la receta (R0). Mide cuánto del desempeño de la receta ya está en la posición continua frente al canal.

In [11]:
m0 = leer("m0_posicion_20261120/resumen_m0.json")
d = m0["desarrollo"]
desarrollo_m0 = pd.DataFrame({
    sistema: {"Mejora media de log-loss frente a B2": f"{r['mejora_b2_media']:.4f}",
              "Log-loss medio": f"{r['logloss_medio']:.4f}", "Brier medio": f"{r['brier_medio']:.4f}",
              "Casillas ganadas a B2": de(r["victorias_b2"], r["casillas"])}
    for sistema, r in (("M0", d["M0_fuera_de_anio"]), ("Receta (R0)", d["R0_fuera_de_anio"]))
}).T
print(f"Desarrollo 2019–2022, fuera de año. C elegido para M0: {d['C_comun_cuatro_anios']}. "
      f"M0 frente a R0 por log-loss: {d['M0_frente_a_R0']['ganadas']} casillas ganadas, "
      f"{d['M0_frente_a_R0']['perdidas']} perdidas.")
desarrollo_m0

Fuente: reports/forecasting/evidence/m0_posicion_20261120/resumen_m0.json
Desarrollo 2019–2022, fuera de año. C elegido para M0: 3.0. M0 frente a R0 por log-loss: 13 casillas ganadas, 11 perdidas.


,Mejora media de log-loss frente a B2,Log-loss medio,Brier medio,Casillas ganadas a B2
M0,0.2311,0.3328,0.1036,21 de 24
Receta (R0),0.2109,0.3145,0.1010,16 de 24


In [12]:
p = m0["prueba"]
prueba_m0 = pd.DataFrame({etiqueta: p["global"][clave] for clave, etiqueta in
                          (("M0", "M0"), ("modelo", "Receta"), ("B2", "B2"))}).T
prueba_m0.columns = ["Log-loss", "Brier", "AP"]
cas = p["casillas"]
print(f"Prueba 2023–2025 (solo reporte), {p['filas']} filas. Casillas de M0: Brier frente a B2 "
      f"{cas['brier_vs_B2']['ganadas']} de {cas['brier_vs_B2']['casillas']}, frente a la receta "
      f"{cas['brier_vs_modelo']['ganadas']} de {cas['brier_vs_modelo']['casillas']}; log-loss frente a B2 "
      f"{cas['logloss_vs_B2']['ganadas']}, frente a la receta {cas['logloss_vs_modelo']['ganadas']}.")
prueba_m0.round(4)

Prueba 2023–2025 (solo reporte), 936 filas. Casillas de M0: Brier frente a B2 11 de 18, frente a la receta 7 de 18; log-loss frente a B2 10, frente a la receta 7.


,Log-loss,Brier,AP
M0,0.4525,0.1494,0.7505
Receta,0.4409,0.1413,0.7651
B2,0.4708,0.1537,0.6115


## 5. Sensibilidad a la regularización

Protocolo declarado antes del cálculo: la rejilla ampliada de 19 valores de C reemplaza a la receta solo si cumple las tres condiciones de la regla. Solo usa semanas objetivo de desarrollo (2019–2022). C = 2 fijo se eligió con los cuatro años, así que su comparación es dentro de muestra.

In [13]:
reg = leer("regularizacion_c_ampliada_20261120/resumen_regularizacion.json")
filas = {"Rejilla ampliada (19 C)": reg["ampliada_fuera_de_anio"], "Receta R0 (6 C)": reg["r0_fuera_de_anio"],
         "C = 2 fijo": reg["C2_fijo"]}
regularizacion = pd.DataFrame({
    sistema: {"Mejora media de log-loss frente a B2": f"{r['mejora_b2_media']:.4f}",
              "Log-loss medio": f"{r['logloss_medio']:.4f}", "Brier medio": f"{r['brier_medio']:.4f}",
              "Casillas ganadas a B2": de(r["victorias_b2"], r["casillas"])}
    for sistema, r in filas.items()
}).T
a, c2 = reg["ampliada_frente_a_r0"], reg["C2_fijo_frente_a_r0"]
regularizacion["Frente a R0: ganadas / perdidas / empates"] = [
    f"{a['ganadas']} / {a['perdidas']} / {a['empates']}", "—", f"{c2['ganadas']} / {c2['perdidas']} / {c2['empates']}"]
print("C fuera de año de la rejilla ampliada:", reg["C_fuera_de_anio_por_anio"],
      "| C común con los cuatro años:", reg["C_comun_cuatro_anios"])
regularizacion

Fuente: reports/forecasting/evidence/regularizacion_c_ampliada_20261120/resumen_regularizacion.json
C fuera de año de la rejilla ampliada: {'2019': 1.5, '2020': 15.0, '2021': 1.0, '2022': 1.0} | C común con los cuatro años: 2.0


,Mejora media de log-loss frente a B2,Log-loss medio,Brier medio,Casillas ganadas a B2,Frente a R0: ganadas / perdidas / empates
Rejilla ampliada (19 C),0.2049,0.3164,0.1013,16 de 24,6 / 6 / 12
Receta R0 (6 C),0.2109,0.3145,0.1010,16 de 24,—
C = 2 fijo,0.2245,0.3110,0.0998,16 de 24,19 / 5 / 0


In [14]:
nombres_regla = {"C_comun_distinto_de_3": "1. El C común con los cuatro años es distinto de 3",
                 "mejora_supera_r0": "2. La mejora media fuera de año supera la de R0",
                 "gana_al_menos_17_de_24_a_r0": "3. Gana al menos 17 de 24 casillas frente a R0"}
regla = pd.Series({nombres_regla[k]: ("se cumple" if v else "no se cumple") for k, v in reg["regla"].items()},
                  name="Regla de reemplazo")
print(f"Reemplaza la receta: {'sí' if reg['reemplaza_receta'] else 'no'}")
regla.to_frame()

Reemplaza la receta: no


,Regla de reemplazo
1. El C común con los cuatro años es distinto de 3,se cumple
2. La mejora media fuera de año supera la de R0,no se cumple
3. Gana al menos 17 de 24 casillas frente a R0,no se cumple


In [15]:
c2c3 = leer("regularizacion_c_ampliada_20261120/c2_frente_a_c3.json")
por_anio = pd.DataFrame([
    {"Año": anio, "Diferencia media de la mejora frente a B2 (C = 2 − C = 3)": v["diferencia_mejora_b2_media"],
     "Casillas ganadas por C = 2 (de 6)": v["casillas_ganadas_c2"]}
    for anio, v in c2c3["por_anio"].items()
]).set_index("Año")
g = c2c3["C2_frente_a_C3"]
print(f"Los cuatro años: diferencia media {c2c3['diferencia_mejora_b2_media']:+.4f}; "
      f"C = 2 gana {g['ganadas']} y pierde {g['perdidas']} de 24 casillas.")
por_anio.round(4)

Fuente: reports/forecasting/evidence/regularizacion_c_ampliada_20261120/c2_frente_a_c3.json
Los cuatro años: diferencia media +0.0009; C = 2 gana 10 y pierde 14 de 24 casillas.


,Diferencia media de la mejora frente a B2 (C = 2 − C = 3),Casillas ganadas por C = 2 (de 6)
Año,,
2019,-0.0008,1
2020,0.0143,5
2021,-0.0053,2
2022,-0.0044,2


## Lectura

Las conclusiones vinculantes son las de la sección 1, con la regla preregistrada. Las secciones 2 a 5 son posteriores a la prueba: describen el comportamiento de la receta y de alternativas, pero no confirman ni sustituyen esas conclusiones, y ninguna se usó para elegir τ, k, W, C, variables ni familia. Hay pocos eventos (10 inicios evaluables por periodo) y las casillas de un mismo municipio están correlacionadas. La discusión completa está en `docs/forecasting/analisis_resultados.md` y `docs/forecasting/analisis_complementario.md`.